# SolarGuard AI — Run all experiments (Colab)
1. Runtime → Change runtime type → **T4 GPU**.
2. Runtime → **Run all**. When asked in step 1, upload `SolarGuard_AI_Final_Code_Package.zip`.
3. At the end a file `results_bundle.zip` downloads. Send it back (it holds `all_results.csv`, `summary_tables.md`, `run_log.txt`, per-class reports).
4. Take screenshots of the final cells' output (log + tables) for the code submission.

In [ ]:
# 1. Upload and unzip the code package
from google.colab import files
up = files.upload()            # choose Project_Code_Text_and_Code_Screenshots.zip
import zipfile, os
zipfile.ZipFile(list(up)[0]).extractall('.')
%cd code
!pip -q install -r requirements.txt

In [ ]:
# 2. Download and extract the public dataset (MIT license)
%cd /content
!git clone -q https://github.com/RaptorMaps/InfraredSolarModules.git
!unzip -q -o InfraredSolarModules/2020-02-14_InfraredSolarModules.zip -d solar_dataset
!ls solar_dataset/InfraredSolarModules | head
%cd /content/code

In [ ]:
# 3. Smoke test, then GPU check
!python solarguard_train.py --smoke-test
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 4. Run all 4 configurations x seeds 42, 43, 44 (log is saved)
!python -u solarguard_train.py --data-root /content/solar_dataset/InfraredSolarModules --output-dir outputs --config all --seeds 42 43 44 --max-epochs 50 2>&1 | tee run_log.txt

In [ ]:
# 5. Paper-ready tables
!python summarize_results.py --output-dir outputs
!cat outputs/duplicate_audit.json

In [ ]:
# 6. Bundle small result files (no model weights) and download
!zip -qr results_bundle.zip outputs run_log.txt -x "*.pt"
files.download('results_bundle.zip')